# Why the vendored ssm CTDS doesn't match an MLE-EM implementation

This notebook isolates and demonstrates the **four M-step discrepancies** in the vendored Jha-lab `ssm` fork (`vendor/Cell_type_dynamical_system/ssm/`) that prevent its `gaussian_ctds` fits from matching a textbook MLE-EM CTDS. Each is shown with a minimal, runnable demonstration.

| # | Issue | Severity |
|---|-------|----------|
| 1 | CVXPY 1.8.1 miscomputes `cp.trace(A @ Var @ B)` (used in the A QP) | **dominant** — A update solves the wrong objective |
| 2 | MAP shrinkage on Q and R instead of MLE | small (priors are tiny by default) |
| 3 | `Q0` is never actually updated (property-getter bug) | structural |
| 4 | Tiny L2 ridge on the A QP + dynamics bias `b` hard-zeroed | negligible / by-design |

The patches that fix 1–4 live in [`../src/pillow_lab_rotation/ssm_patches.py`](../src/pillow_lab_rotation/ssm_patches.py).


In [5]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import cvxpy as cp
import ssm.lds as ssm_lds
from adithis_utils.simulation_utils import create_dynamics_matrix

print('cvxpy version:', cp.__version__)


cvxpy version: 1.8.1


## 1. The CVXPY `cp.trace` bug (the big one)

ssm's constrained-A M-step ([`observations.py:1319`](../vendor/Cell_type_dynamical_system/ssm/ssm/observations.py)) builds the QP objective with a term

```python
objective = cp.Minimize(cp.quad_form((L.T@W).flatten(), kron_ExuxuTs)
                        - 2*cp.trace(Q_inv @ W @ (ExuyTs_k + self.h0_k)))
```

The intent is `tr(Q⁻¹ W M_Δ)`. But in **CVXPY 1.8.1, `cp.trace(A @ Var @ B)` does not return the matrix trace** — it returns the Frobenius inner product `sum((A @ Var) * B)`. So the A update is optimizing a different objective than the one written in the math.

### 1a. Minimal reproduction


In [6]:
np.random.seed(0)
Dm = 3
W = cp.Variable((Dm, Dm))
W.value = np.random.randn(Dm, Dm)
A = np.random.randn(Dm, Dm)
B = np.random.randn(Dm, Dm)

true_trace      = np.trace(A @ W.value @ B)
cp_trace        = cp.trace(A @ W @ B).value
frobenius_inner = ((A @ W.value) * B).sum()
correct_workaround = cp.sum(cp.diag(A @ W @ B)).value

print(f'np.trace(A @ W @ B)          = {true_trace:.6f}   <- correct')
print(f'cp.trace(A @ W @ B)          = {cp_trace:.6f}   <- WRONG (CVXPY 1.8.1)')
print(f'sum((A @ W) * B)             = {frobenius_inner:.6f}   <- what cp.trace actually returns')
print(f'cp.sum(cp.diag(A @ W @ B))   = {correct_workaround:.6f}   <- correct workaround')
print()
print(f'cp.trace matches Frobenius inner product? {np.isclose(cp_trace, frobenius_inner)}')
print(f'cp.trace matches true trace?              {np.isclose(cp_trace, true_trace)}')


np.trace(A @ W @ B)          = -12.826757   <- correct
cp.trace(A @ W @ B)          = 5.741892   <- WRONG (CVXPY 1.8.1)
sum((A @ W) * B)             = 5.741892   <- what cp.trace actually returns
cp.sum(cp.diag(A @ W @ B))   = -12.826757   <- correct workaround

cp.trace matches Frobenius inner product? True
cp.trace matches true trace?              False


Note this only bites for **full** matrices. If `A @ W @ B` happens to be diagonal (e.g. the identity-weighted case), the Frobenius inner product and the trace coincide and the bug is invisible — which is probably why it went unnoticed.

### 1b. Downstream impact: the unconstrained A QP doesn't recover OLS

The dynamics A update should, with no sign constraints, reproduce the ordinary least-squares solution $A = M_\Delta^\top M_{1:T-1}^{-1}$. We feed ssm's `_solve_constrained_A` consistent statistics and check.


In [7]:
# Generate consistent dynamics statistics from a known Dale's-law A.
np.random.seed(42)
D, De, Di = 5, 2, 3
list_of_dimensions = np.array([[De, Di]])

A_true = np.zeros((D, D))
A_true[:, :De] = np.random.rand(D, De) * 0.5
A_true[:, De:] = -np.random.rand(D, Di) * 0.5
A_true[np.diag_indices(D)] = np.random.rand(D) - 0.5

np.random.seed(1)
T = 2000
x = np.zeros((T, D))
chol = np.linalg.cholesky(np.eye(D) * 0.01)
for t in range(1, T):
    x[t] = A_true @ x[t - 1] + chol @ np.random.randn(D)

M1Tm1  = x[:-1].T @ x[:-1]        # sum E[x_{t-1} x_{t-1}^T]
M_delta = x[:-1].T @ x[1:]        # sum E[x_{t-1} x_t^T]
Q = np.eye(D) * 0.01

A_ols = M_delta.T @ np.linalg.inv(M1Tm1)
print('OLS A (unconstrained MLE) — diagonal:', np.round(np.diag(A_ols), 3))


OLS A (unconstrained MLE) — diagonal: [ 0.278 -0.32   0.051  0.114 -0.451]


In [4]:
# Build a throwaway ssm CTDS dynamics object so we can call _solve_constrained_A.
emission_kwargs = dict(cell_identity=np.array([1]*5 + [2]*5),
                       region_identity=np.zeros(10, dtype=int),
                       list_of_dimensions=list_of_dimensions)
dynamics_kwargs = dict(list_of_dimensions=list_of_dimensions)
m = ssm_lds.LDS(N=10, D=D, M=0, dynamics='gaussian_ctds', emissions='gaussian_ctds',
               emission_kwargs=emission_kwargs, dynamics_kwargs=dynamics_kwargs)
m.dynamics.J0_k = np.zeros((D, D))   # no ridge, so we isolate the trace bug
m.dynamics.h0_k = np.zeros((D, D))

A_ssm = m.dynamics._solve_constrained_A(0, M1Tm1, M_delta, Q)
print('ssm _solve_constrained_A — diagonal:', np.round(np.diag(A_ssm), 3))
print()
print(f'Frobenius distance  ssm vs OLS: {np.linalg.norm(A_ssm - A_ols):.4f}')
print(f'(constraints are satisfied by OLS here, so a correct QP would return OLS)')


Assuming Dale's constraints for the weights within regions
Assuming FOF-ADS cross-region constraints if 2 regions, otherwise no cross-region constraints
ssm _solve_constrained_A — diagonal: [ 0.088 -0.31   0.032  0.074 -0.325]

Frobenius distance  ssm vs OLS: 0.9670
(constraints are satisfied by OLS here, so a correct QP would return OLS)


In [5]:
# Now patch _solve_constrained_A to route through cp.sum(cp.diag(...)) and re-solve.
import types

def fixed_solve_constrained_A(self, k, ExuxuTs_k, ExuyTs_k, Sigmas_k):
    D, M, lags = self.D, self.M, self.lags
    Q_inv = np.linalg.inv(Sigmas_k); Q_inv = Q_inv / np.max(np.abs(Q_inv))
    L = np.linalg.cholesky(Q_inv)
    kron_ExuxuTs = np.kron((ExuxuTs_k + self.J0_k).T, np.eye(D))
    W = cp.Variable((D, D*lags + M))
    constraints = self.within_region_constraints(W) + self.across_region_constraints(W)
    objective = cp.Minimize(
        cp.quad_form((L.T @ W).flatten(order='F'), cp.psd_wrap(kron_ExuxuTs))
        - 2 * cp.sum(cp.diag(Q_inv @ W @ (ExuyTs_k + self.h0_k)))   # <- cp.trace -> cp.sum(cp.diag)
    )
    cp.Problem(objective, constraints).solve(solver=cp.MOSEK, verbose=False)
    return W.value

m.dynamics._solve_constrained_A = types.MethodType(fixed_solve_constrained_A, m.dynamics)
A_fixed = m.dynamics._solve_constrained_A(0, M1Tm1, M_delta, Q)

print('patched _solve_constrained_A — diagonal:', np.round(np.diag(A_fixed), 3))
print()
print(f'Frobenius distance  patched vs OLS: {np.linalg.norm(A_fixed - A_ols):.2e}')
print('=> the cp.sum(cp.diag(...)) workaround recovers the correct A.')


patched _solve_constrained_A — diagonal: [ 0.278 -0.32   0.051  0.114 -0.451]

Frobenius distance  patched vs OLS: 7.84e-08
=> the cp.sum(cp.diag(...)) workaround recovers the correct A.


### 1c. Is this fixed in a newer cvxpy?

Yes. This is a **cvxpy regression**, confirmed by testing the repro across versions (the bug is purely in expression evaluation — `.value` — so no solver is needed to reproduce it):

| cvxpy | `cp.trace(A @ W @ B)` correct? | ssm-style A QP recovers OLS? |
|-------|-------------------------------|------------------------------|
| 1.6.0 | yes | — |
| 1.7.0 | yes | — |
| **1.8.0** | **no — bug introduced** | no (err 1.46 from OLS) |
| 1.8.1 (this env) | no | no (err 1.46) |
| **1.9.0** | **yes — fixed** | yes (err 2.6e-16) |

Under cvxpy >= 1.9.0 the ssm A QP recovers OLS exactly with no code change. So issue #1 can be resolved either by the `cp.sum(cp.diag(...))` workaround (works on any version) **or** by upgrading cvxpy to >= 1.9.0.

Issues #2-#4 below are ssm design choices (MAP priors, the Q0 property bug, hard-zeroing `b`), **not** cvxpy bugs, so they still need the patches in `ssm_patches.py` for MLE parity regardless of cvxpy version.

**Upstream provenance (cvxpy GitHub):**

- Introduced by [PR #2920](https://github.com/cvxpy/cvxpy/pull/2920) "Implement Trace(AB) using vdot for performance efficiency" (merged 2025-08-30, commit `ad963b6`). It rewrote `trace(A @ B)` to `vdot(args[0], args[1])` for a ~7-10x speedup, but `vdot` computes the *conjugate* inner product = trace(A^H @ B), not trace(A @ B). For real matrices that's the Frobenius inner product `sum(A * B)` — exactly what we see above.
- Fixed by [PR #3196](https://github.com/cvxpy/cvxpy/pull/3196) "Fix silent wrong objective in `cp.trace(A @ B)` for non-symmetric matrices" (merged 2026-03-07, commit `25b39e8`). Replaces `vdot(args[0], args[1])` with `sum(multiply(args[0], args[1].T))` — adding the missing transpose. Backported to the `patch/1.8.2` branch on 2026-03-17.

So issue #1 can be resolved by the `cp.sum(cp.diag(...))` workaround (any version), or by upgrading cvxpy to **1.8.2** (patch-level) or **1.9.0**.


## 2. MAP shrinkage on Q and R (vs MLE)

ssm estimates the noise covariances with a MAP estimate under an inverse-Wishart prior, not the MLE. From [`regression.py:222`](../vendor/Cell_type_dynamical_system/ssm/ssm/regression.py) (emissions R) and [`observations.py:1373`](../vendor/Cell_type_dynamical_system/ssm/ssm/observations.py) (dynamics Q):

```python
Sigma = (expected_err + Psi0 * np.eye(d)) / (nu0 + weight_sum + d + 1)
```

with defaults `Psi0 = 1e-4`, `nu0 = 1e-4`. A textbook MLE uses `expected_err / weight_sum`. The prior is tiny, so the effect is small for large datasets — but the two estimators are not identical, and the denominator differs by `nu0 + d + 1`.


In [6]:
# Demonstrate the gap on a small residual-error matrix.
np.random.seed(0)
d_dim = 6
weight_sum = 200          # e.g. n_trials * T
err = np.random.randn(d_dim, d_dim); expected_err = err @ err.T   # PSD "sum of squared residuals"

Psi0, nu0 = 1e-4, 1e-4
Sigma_map = (expected_err + Psi0 * np.eye(d_dim)) / (nu0 + weight_sum + d_dim + 1)
Sigma_mle = expected_err / weight_sum

print('MAP diag (ssm):', np.round(np.diag(Sigma_map), 5))
print('MLE diag      :', np.round(np.diag(Sigma_mle), 5))
print(f'max abs diff: {np.abs(Sigma_map - Sigma_mle).max():.3e}')
print(f'denominators: MAP = {nu0 + weight_sum + d_dim + 1}, MLE = {weight_sum}')
print()
print('With fewer samples the gap grows — e.g. weight_sum = 20:')
ws = 20
print('  MAP/MLE diag ratio:', np.round(np.diag((expected_err + Psi0*np.eye(d_dim))/(nu0+ws+d_dim+1)) /
                                        np.diag(expected_err/ws), 4))


MAP diag (ssm): [0.06616 0.01565 0.01535 0.04382 0.05707 0.02427]
MLE diag      : [0.06847 0.0162  0.01588 0.04535 0.05906 0.02512]
max abs diff: 2.315e-03
denominators: MAP = 207.0001, MLE = 200

With fewer samples the gap grows — e.g. weight_sum = 20:
  MAP/MLE diag ratio: [0.7407 0.7408 0.7408 0.7407 0.7407 0.7408]


## 3. `Q0` (initial-state covariance) is never updated

In ssm, `Sigmas_init` is exposed as a **`@property`** ([`lds.py:973`](../vendor/Cell_type_dynamical_system/ssm/ssm/lds.py)) that reconstructs a fresh array from `_sqrt_Sigmas_init` on every access:

```python
@property
def Sigmas_init(self):
    return np.matmul(self._sqrt_Sigmas_init, np.swapaxes(self._sqrt_Sigmas_init, -1, -2))
```

The dynamics M-step then does `self.Sigmas_init[k] = <new Q0>` ([`observations.py:1129`](../vendor/Cell_type_dynamical_system/ssm/ssm/observations.py)) — but that indexes into the **throwaway array returned by the getter**, so the assignment is silently discarded. `Q0` stays at whatever it was initialized to.


In [7]:
D = 5
list_of_dimensions = np.array([[2, 3]])
emission_kwargs = dict(cell_identity=np.array([1]*5 + [2]*5),
                       region_identity=np.zeros(10, dtype=int),
                       list_of_dimensions=list_of_dimensions)
dynamics_kwargs = dict(list_of_dimensions=list_of_dimensions)
m = ssm_lds.LDS(N=10, D=D, M=0, dynamics='gaussian_ctds', emissions='gaussian_ctds',
               emission_kwargs=emission_kwargs, dynamics_kwargs=dynamics_kwargs)

print('Sigmas_init[0] before:'); print(np.round(m.dynamics.Sigmas_init[0], 2))

# This is exactly what the M-step does — assign into the property's return value.
m.dynamics.Sigmas_init[0] = np.eye(D) * 999.0

print()
print('Sigmas_init[0] after `Sigmas_init[0] = 999*I`:')
print(np.round(m.dynamics.Sigmas_init[0], 2))
print()
print('=> unchanged. The assignment mutated a temporary array and was discarded.')
print()
print('The setter-based path *does* stick:')
m.dynamics.Sigmas_init = np.array([np.eye(D) * 7.0])
print('Sigmas_init[0] diag after using the setter:', np.round(np.diag(m.dynamics.Sigmas_init[0]), 2))
print()
print('Correct fix in the M-step: write through _sqrt_Sigmas_init directly:')
m.dynamics._sqrt_Sigmas_init[0] = np.linalg.cholesky(np.eye(D) * 4.0)
print('Sigmas_init[0] diag:', np.round(np.diag(m.dynamics.Sigmas_init[0]), 2))


Assuming Dale's constraints for the weights within regions
Assuming FOF-ADS cross-region constraints if 2 regions, otherwise no cross-region constraints
Sigmas_init[0] before:
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]

Sigmas_init[0] after `Sigmas_init[0] = 999*I`:
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]

=> unchanged. The assignment mutated a temporary array and was discarded.

The setter-based path *does* stick:
Sigmas_init[0] diag after using the setter: [7. 7. 7. 7. 7.]

Correct fix in the M-step: write through _sqrt_Sigmas_init directly:
Sigmas_init[0] diag: [4. 4. 4. 4. 4.]


## 4. L2 ridge on A, and the dynamics bias `b` is hard-zeroed

Two smaller items, shown from the source:

**(a) L2 ridge.** The dynamics object builds `J0` / `h0` ridge terms ([`observations.py`](../vendor/Cell_type_dynamical_system/ssm/ssm/observations.py)) with `l2_penalty_A = 1e-8` and adds them into the A QP. Negligible at default strength, but it means the A update is a (very lightly) penalized regression, not pure MLE.

**(b) `b` hard-zeroed.** The dynamics M-step assigns `bs[k] = np.zeros(D)` every iteration ([`observations.py:1367`](../vendor/Cell_type_dynamical_system/ssm/ssm/observations.py)). So even though `gaussian_ctds` carries a dynamics-bias parameter, ssm never fits it — it is forced to zero on every M-step. To match, the local CTDS uses `fit_b=False`.


In [8]:
import inspect
from ssm.observations import AutoRegressiveCellTypeObservations

src = inspect.getsource(AutoRegressiveCellTypeObservations.m_step)
# Show the lines that hard-zero b and form the MAP Sigma.
for line in src.splitlines():
    s = line.strip()
    if 'bs[k]' in s or 'Sigmas[k] = ' in s or 'nu0' in s:
        print(line)


            bs[k] = np.zeros(D) 
            nu = self.nu0 + Ens[k]
            Sigmas[k] = (sqerr + self.Psi0) / (nu + D + 1) 
                bs[k] = bs[i] + 0.01 * npr.randn(*bs[i].shape)
                Sigmas[k] = Sigmas[i]


In [9]:
# Confirm empirically: after an M-step, bs[0] is exactly zero regardless of the data.
np.random.seed(0)
N = 10; De, Di = 2, 3; D = 5
A_true = create_dynamics_matrix(np.array([[De, Di]]))
C_true = np.zeros((N, D)); C_true[:5, :De] = np.random.rand(5, De); C_true[5:, De:] = np.random.rand(5, Di)
Q_true = np.eye(D) * 0.01
R_true = np.diag(np.random.rand(N) + 0.1) / 100   # well-conditioned observation noise

emission_kwargs = dict(cell_identity=np.array([1]*5 + [2]*5), region_identity=np.zeros(N, dtype=int),
                       list_of_dimensions=np.array([[De, Di]]))
dynamics_kwargs = dict(list_of_dimensions=np.array([[De, Di]]))
m = ssm_lds.LDS(N=N, D=D, M=0, dynamics='gaussian_ctds', emissions='gaussian_ctds',
               emission_kwargs=emission_kwargs, dynamics_kwargs=dynamics_kwargs)
m.dynamics.As[0] = A_true
m.dynamics._sqrt_Sigmas[0] = np.linalg.cholesky(Q_true)
m.emissions.Cs[0] = C_true
m.emissions.inv_etas[0] = R_true       # inv_etas IS the covariance for GaussianCellTypeEmissions

# Sample data from these (valid) params, then deliberately set b nonzero.
datas = [m.sample(80)[1] for _ in range(10)]
m.dynamics.bs[0] = np.random.randn(D)
print('bs[0] set to:', np.round(m.dynamics.bs[0], 3))

m.fit(datas, method='laplace_em', num_iters=1, initialize=False, verbose=0)
print('bs[0] after one EM step:', np.round(m.dynamics.bs[0], 3), '<- forced to zero')


Assuming Dale's constraints for the weights within regions
Assuming FOF-ADS cross-region constraints if 2 regions, otherwise no cross-region constraints
bs[0] set to: [-0.183 -1.178  1.529  0.287  1.099]
bs[0] after one EM step: [0. 0. 0. 0. 0.] <- forced to zero


## Summary

| # | Issue | Fix in `ssm_patches.py` |
|---|-------|--------------------------|
| 1 | `cp.trace(A @ Var @ B)` returns Frobenius inner product | route through `cp.sum(cp.diag(...))` |
| 2 | MAP shrinkage on Q, R | `Psi0=0`, `nu0=-N-1` → denominator = `weight_sum` (MLE) |
| 3 | `Q0` never updates (property bug) | write through `_sqrt_Sigmas_init`, averaging over trials |
| 4a | L2 ridge on A | zero `J0_k`, `h0_k` |
| 4b | `b` hard-zeroed | match with local `fit_b=False` |

With these four fixes applied via `patch_ssm_ctds(model)`, the patched ssm CTDS and the local MLE CTDS produce **identical LL trajectories and fitted parameters** (Fro distance ~1e-3) when started from the same point — see [`testing_against_aditis.ipynb`](testing_against_aditis.ipynb) §3a–§5b.
